In [ ]:
import os
import zarr
import numpy as np
import dask.array as da
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib import colors, patches
import matplotlib.image as mpimg
from sklearn.model_selection import train_test_split

CLASS_MERING = True # TRUE: Binary classification, FALSE: Multiclass classification

In [ ]:
raw_data_dir = '/home/laura/Scriptie/ruwe_data'
bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/PhiSat2'

os.makedirs(bewerkte_data_dir, exist_ok=True)

path_phiSat2 = os.path.join(raw_data_dir, 'burned.zarr.zip')

MAX_PATCHES = 50

def patches_complete(patch_keys, group):
    all_X = []
    all_y = []

    for key in patch_keys:
        patch = group[key]
        
        img_array = patch['img'][:]
        label_array = patch['label'][:]
        
        img_reshaped = np.transpose(img_array, (1, 2, 0))
        label_index = np.argmax(label_array, axis=0)
        
        X_flat = img_reshaped.reshape(-1, 7) 
        y_flat = label_index.reshape(-1)
        
        all_X.append(X_flat)
        all_y.append(y_flat)

    X_dataset = np.vstack(all_X)
    y_dataset = np.concatenate(all_y)

    if CLASS_MERING:
        y_dataset = (y_dataset == 1).astype(np.uint8)
    else:
        print("CLASS_MERGING is set to False. Using multiclass labels.")

    return X_dataset, y_dataset

try:
    store = zarr.ZipStore(path_phiSat2, mode='r')
    zarr_root = zarr.group(store=store)
    trainval_group = zarr_root['trainval']
    
    patch_keys = list(trainval_group.keys())[:MAX_PATCHES]

    train_keys, val_keys = train_test_split(patch_keys, test_size=0.2, random_state=42)

    X_train, y_train = patches_complete(train_keys, trainval_group)
    X_val, y_val = patches_complete(val_keys, trainval_group)

    np.save(os.path.join(bewerkte_data_dir, 'X_train_flat.npy'), X_train)
    np.save(os.path.join(bewerkte_data_dir, 'y_train_flat.npy'), y_train)

    np.save(os.path.join(bewerkte_data_dir, 'X_val_flat.npy'), X_val)
    np.save(os.path.join(bewerkte_data_dir, 'y_val_flat.npy'), y_val)

    print(f" - Train matrix : {X_train.shape}")
    print(f" - Val matrix   : {X_val.shape}")

    store.close()

except Exception as e:
    print(f"Error: {e}")